# ETF 定期定額 + 低檔加碼策略（左側交易）互動式研究與參數最佳化

本 Notebook 提供：
1. **單一策略回測與互動式圖表**（三連動圖、水下圖與交易明細）
2. **參數網格最佳化 (Grid Search)**（尋找最佳比較天數、跌幅門檻與加碼倍數）
3. **參數敏銳度分析**（排除過度擬合孤立點，尋找穩健高原區）

## 1. 載入模組與歷史資料

In [ ]:
import sys
import pandas as pd
import plotly.express as px
sys.path.append('..')

from src.data_loader import DataLoader
from src.dca_addon.config import StrategyConfig, AddOnRule
from src.dca_addon.engine import DCAAddOnBacktester
from src.dca_addon.metrics import evaluate_performance
from src.dca_addon.optimizer import grid_search
from src.dca_addon.visualizer import plot_dca_summary, generate_html_report

# 載入行情數據 (以 0050.TW 為例)
loader = DataLoader()
df = loader.fetch_data('33YTW.EZ', start_date='2022-01-01', end_date='2026-07-30')
print(f"成功載入 {len(df)} 筆 K 線資料，區間: {df.index[0].date()} ~ {df.index[-1].date()}")

## 2. 單一策略回測與互動視覺化

In [ ]:
# 設定基礎定期定額與低檔加碼規則
config = StrategyConfig(
    base_invest_amount=100,
    rules=[
        AddOnRule(lookback_days=20, drawdown_threshold=0.05, multiplier=2),
        AddOnRule(lookback_days=20, drawdown_threshold=0.20, multiplier=2.0)
    ],
    cooldown_days=0,
    max_addons_per_month=7,
    total_extra_budget=50000000
)

bt = DCAAddOnBacktester(df, config)
equity, trades = bt.run()
metrics = evaluate_performance(equity, trades, config)

print("=== 回測績效摘要 ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

# 繪製 Plotly 互動式三連動圖表
fig = plot_dca_summary(df, equity, trades)
fig.show()

## 3. 參數最佳化網格搜尋 (Grid Search Optimization)

In [ ]:
# 定義要搜尋的參數網格空間
param_grid = {
    'lookbacks': [10],                  # 比較天數
    'first_thresholds': [0.05, 0.06, 0.1],         # 第一階梯跌幅 (5%, 6%, 8%)
    'second_thresholds': [0.10, 0.15, 0.20],        # 第二階梯跌幅 (10%, 12%, 15%)
    'first_multipliers': [1.5, 2.0],                # 第一階梯加碼倍數
    'second_multipliers': [2.0, 3.0],                # 第二階梯加碼倍數
    'cooldowns': [0],                         # 冷卻天數
    'monthly_limits': [3, 5, 7, 20]                        # 每月加碼次數上限
}

print("開始執行網格掃描中...")
results_df = grid_search(df, param_grid, base_amount=100, total_extra_budget=500000)
results_df = results_df.sort_values('total_return', ascending=False).reset_index(drop=True)

print(f"完成！共測試 {len(results_df)} 組參數組合。")
# 顯示前 10 名最佳參數組合
display_cols = ['lookback', 't1', 't2', 'm1', 'm2', 'cooldown', 'monthly_limit', 'total_return', 'cagr', 'max_drawdown', 'alpha_vs_pure_dca', 'addon_count']
results_df[display_cols].head(10)

## 4. 參數敏感度與穩健性熱力圖 (Parameter Sensitivity Heatmap)

In [ ]:
# 以 比較天數 (lookback) x 第一階梯跌幅 (t1) 繪製 Alpha 超額收益熱力圖
heatmap_df = results_df.groupby(['lookback', 't1'])['alpha_vs_pure_dca'].mean().reset_index()
pivot_df = heatmap_df.pivot(index='lookback', columns='t1', values='alpha_vs_pure_dca')

fig_heat = px.imshow(
    pivot_df,
    labels=dict(x="第一階梯跌幅門檻 (t1)", y="比較天數 (Lookback)", color="Alpha (超額收益)"),
    x=[f"{t:.0%}" for t in pivot_df.columns],
    y=[f"{lb}天" for lb in pivot_df.index],
    text_auto=".2%",
    color_continuous_scale="Viridis",
    title="參數高原敏銳度熱力圖 (Lookback vs 跌幅門檻對超額收益之影響)"
)
fig_heat.show()